<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🤖 ReAct Agent —— 手写 vs 一行预构建

**一句话定位**:ReAct = **Act(调工具) → Observe(看结果) → Reason(再决定)** 循环;手写要 7 步,`create_agent` 一次搞定。

</div>

### 1️⃣ ReAct 循环的本质

```
START → assistant ──有 tool_call──▶ tools ──┐
            ▲                              │
            └──────────────────────────────┘
            │
            └──无 tool_call──▶ END
```

| 步骤 | 做什么 |
|---|---|
| **Reason** | `assistant` 节点:LLM 看全部 `messages`,决定直接答 or 调工具 |
| **Act** | `ToolNode`:执行 `AIMessage.tool_calls`,产出 `ToolMessage` |
| **Observe** | 工具结果回到 `messages`,回到 `assistant` 再判断 |
| **终止** | 最新 `AIMessage` 没有 `tool_calls` → 结束 |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ✋ 第一部分:手写版(理解原理用)

</div>

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b
def multiply(a: int, b: int) -> int:
    """Multiply a and b."""
    return a * b
tools = [add, multiply]

llm_with_tools = ChatOpenAI(model="gpt-4o").bind_tools(tools, parallel_tool_calls=False)   # ①
sys_msg = SystemMessage("You are a helpful arithmetic assistant.")

def assistant(state: MessagesState):                                                        # ②
    return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

builder = StateGraph(MessagesState)                                                         # ③
builder.add_node("assistant", assistant)                                                    # ④
builder.add_node("tools", ToolNode(tools))                                                  # ⑤
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)                                 # ⑥
builder.add_edge("tools", "assistant")                                                      # ⑦ 闭环
react_graph = builder.compile()

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ⚡ 第二部分:预构建版(日常用这个)

</div>

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-4o",          # 字符串即可
    tools=[add, multiply],          # 普通函数 / @tool 均可
    system_prompt="You are a helpful arithmetic assistant.",
)

result = agent.invoke({"messages": [("human", "Add 3 and 4, then multiply the result by 2")]})
for m in result["messages"]:
    m.pretty_print()

#### 手写部分 ↔ 预构建参数

| 手写 | `create_agent` |
|---|---|
| `bind_tools` + `assistant` 节点 | `model` + `tools` |
| `SystemMessage` 拼接 | `system_prompt` |
| `ToolNode` + `tools_condition` + 两条边 + `compile()` | 内置,返回已编译的图 |
| `MessagesState` | 默认 state(自带 `messages`) |
| 扩展 state | `state_schema=` |
| 记忆 / 持久化 | `checkpointer=` · `store=` |
| 人工审批 | `interrupt_before=` · `interrupt_after=` |
| 结构化输出 | `response_format=` |
| 钩子(裁剪/摘要/护栏) | `middleware=` |

返回值是 **compiled graph**:`invoke` · `stream` · `get_graph().draw_mermaid_png()` 用法与手写版完全一致。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:`create_agent` 不是新概念,就是把「`assistant` + `ToolNode` + `tools_condition` + 回环」打包。**能手写 ≠ 需要手写**——除非要在循环里插自定义节点/分支。

</div>

### 🧭 版本与选择

| API | 状态 |
|---|---|
| `langchain.agents.create_agent` | ✅ LangChain v1+ **推荐**(本机 langchain 1.3.1 可用) |
| `langgraph.prebuilt.create_react_agent` | ⚠️ 旧版,参数为 `prompt=`;v1 起建议迁移 |
| 手写 `StateGraph` | 🔬 学原理 / 需要自定义拓扑(多 agent、路由、审批) |

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

💡 **注意**:手写版里用 `bind_tools(..., parallel_tool_calls=False)` 关闭并行工具调用;用 `create_agent` 时没有这个入口,需要时可传一个**已经 bind 好的 model 实例**,或改用 middleware / 查文档——此点尚未实测。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ✨ 一句话带走

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**ReAct = LLM 节点 ⇄ ToolNode 的循环,以 "最新消息有无 `tool_calls`" 作为出口**;手写是为了懂它,**日常用 `create_agent(model, tools, system_prompt)`** 三个参数搞定。

</div>